In [1]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

import logging

In [2]:
# Generar una funcion que obtenga el modelo LLM

def get_llm(prefix_model, temperature=0.9):
    """
    Obtiene un modelo de lenguaje (LLM) basado en el modelo especificado.

    Args:
        model (str): El nombre del modelo a utilizar.
        temperature (float, optional): La temperatura para la generación de texto. 
                                        Valores más altos producen respuestas más creativas.
                                        Valores más bajos producen respuestas más deterministas.
                                        Por defecto es 0.9.

    Returns:
        llm: Un objeto de modelo de lenguaje inicializado.
    """
    llm = init_chat_model(
                            model=prefix_model, 
                            # temperature=temperature
                          )
    return llm

In [3]:
# Configuración del modelo de lenguaje
load_dotenv()

prefix_model = os.getenv("PREFIX_MODEL")

# Obtención del modelo
llm = get_llm(prefix_model)


### Fundamentos de RAG

| Estrategia | Caracteristicas | Limitaciones |
|--|--|--|
| Contexto | Incluir todo la información de los documentos en el PROMPT | Ventana del contexto |
| Fine tuning | Reentranar el modelo, atado a la infraestructura | Tiempo de actualización |
| RAG | Particiona el docuemnto en fragmentos y coloca los fragmentos en el PROMPT | Tratamiento de los documentos |


In [4]:
# Consulta sin tener la informacion de la politica de devolucion de la empresa TiendaYa

PREGUNTA = "¿Cuál es la política de devolución de la empresa TiendaYa para productos en oferta?"

respuesta = llm.invoke(PREGUNTA)

print("Respuesta: " + respuesta.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Respuesta: Como modelo de inteligencia artificial, no tengo acceso a información en tiempo real ni a bases de datos internas de empresas específicas llamadas "TiendaYa" (ya que puede haber varios comercios con ese nombre en diferentes países). 

Para conocer la política de devolución exacta para productos en oferta de **TiendaYa**, te recomiendo hacer lo siguiente:

1. **Revisar su sitio web oficial:** Busca en el pie de página (footer) secciones como *"Políticas de devolución"*, *"Términos y condiciones"* o *"Preguntas frecuentes (FAQ)"*.
2. **Contactar a su servicio al cliente:** Escríbeles a través de su chat en vivo, WhatsApp de soporte o correo electrónico oficial.

*Nota general sobre el comercio electrónico:* Muchas tiendas suelen tener políticas más restrictivas con los productos en oferta o en liquidación (a veces llamados "venta final"), por lo que siempre es aconsejable verificarlo antes de realizar la compra.


### Aplicando RAG

- 1. Carga y procesamiento del documento. ( INGESTA DE DATOS)
- 2. Realizar el embedding usando modelos de IA. ( INGESTA DE DATOS)
- 3. Uso de base de datos vectorial ( INGESTA DE DATOS)
- 4. Realización de consultas

#### 1. Carga y procesamiento del documento

In [5]:
# Librería pypdf

from pypdf import PdfReader
from langchain_core.documents import Document

ARCHIVO = "data/politica_tiendaya.pdf"

reader = PdfReader(ARCHIVO)

documentos = []

for i, pagina in enumerate(reader.pages):
    
    texto = pagina.extract_text()
    print(f"Página {i+1}: {texto[:100]}...")  # Muestra los primeros 100 caracteres de cada página

    doc = Document(page_content=texto, 
                   metadata={"pagina": i+1})
    
    documentos.append(doc)

print("Número de páginas procesadas: ", len(documentos))


Página 1: TiendaYa — Política de devoluciones y reembolsos
Versión 2.1 · Documento ficticio para uso en el cur...
Página 2: 5. Productos dañados en el envío
Si el producto llega dañado, el cliente debe reportarlo dentro de l...
Página 3: 8. Cómo iniciar una devolución
El cliente debe ingresar a la sección Mis pedidos, seleccionar el ped...
Número de páginas procesadas:  3


In [6]:
# Segmentacion de documentos

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Se va a segmentar el documento en fragmetos con solapamientos

# Dividir el fragmentos de 500 carateres con un solapamiento de 50 caracteres

chunk_size = 500
chunk_overlap = 50

spliter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,  # fragmento de 500 caracteres
    chunk_overlap=chunk_overlap # solapamiento de 50 caracteres
)

fragmentos = spliter.split_documents(documentos)

print(f"{'chunk_size':>10} {'overlap':>8} {'fragmentos':>11}")
print(f"{chunk_size:>10} {chunk_overlap:>8} {len(fragmentos):>11}")

chunk_size  overlap  fragmentos
       500       50           7


#### 2. Realizar el embedding usando modelos de IA. ( INGESTA DE DATOS)

Un modelo de embeddings convierte un texto en un vector de números. Textos con significado similar producen vectores cercanos; la cercanía se mide con la **similitud coseno** (1 = idénticos, 0 = sin relación). Es el mismo principio del Módulo 1, ahora con modelos entrenados para este fin.

| Modelo | Proveedor | Ejecución | Consideraciones | Dimensiones |
|---|---|---|---|--|
| BGE (bge-m3) | BAAI, abierto | Local, vía Ollama | Multilingüe; sin costo; los datos no salen del equipo |1,024|
| text-embedding-3 | OpenAI | API | Alta calidad; costo por token |3,072|
| gemini-embedding-001 | Gemini | API | Google |3,072|

El modelo de embeddings usado al indexar debe ser **el mismo** que se use al consultar: vectores de modelos distintos no son comparables.

In [7]:
import math 

def similitud_coseno(v1, v2):
    """
    SIMILITUD COSENO = mide el angulo entre dos vectores.
    1.0  -> apuntan exactamente igual (muy parecidos)
    0.0  -> no tienen relacion
    Es la metrica que usan los LLMs reales para comparar embeddings.
    """
    producto_punto = sum(a * b for a, b in zip(v1, v2))
    norma1 = math.sqrt(sum(a * a for a in v1))
    norma2 = math.sqrt(sum(b * b for b in v2))
    if norma1 == 0 or norma2 == 0:
        return 0.0
    return producto_punto / (norma1 * norma2)

In [8]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

MODEL_EMBEDDINGS = "gemini-embedding-001"  # Modelo de embeddings de Google GenAI

embeddings = GoogleGenerativeAIEmbeddings(model=MODEL_EMBEDDINGS)  # Modelo de embeddings de Google GenAI


In [9]:
PREGUNTA = "¿Cuál es la política de devolución de la empresa TiendaYa para productos en oferta?"

# Generar el vector de la pregunta
vector_pregunta = embeddings.embed_query(PREGUNTA)

print("Vector de la pregunta: ", vector_pregunta[:20], "...")  # Muestra los primeros 200 valores del vector

Vector de la pregunta:  [0.014368168, -0.007729253, -0.011264546, -0.06577767, -0.00772031, 0.038819022, 0.0022101377, 0.025668778, -0.01367702, -0.01004288, 0.004157194, 0.02342531, 0.0075691566, -0.027637, 0.11160647, -0.00017570842, 0.022733022, 0.006427428, -0.009605082, -0.0072372607] ...


In [10]:
'''
for fragmento in fragmentos:
    print(fragmento.page_content[20], "...")  # Muestra los primeros 100 caracteres del fragmento
'''

vector_fragmentos = [ fragmento.page_content for fragmento in fragmentos]

#print(vector_fragmentos[:5])  # Muestra los primeros 5 fragmentos

vector_fragmentos_embeddings = embeddings.embed_documents(vector_fragmentos)

# print("Vector de embeddings de los fragmentos: ", vector_fragmentos_embeddings[:20], "...")  # Muestra los primeros 200 valores del vector

In [11]:
# Buscando similitudes entre el vector de la pregunta y los vectores de los fragmentos
similitudes = []

for i, fragmento in enumerate(vector_fragmentos_embeddings):
    similitud = similitud_coseno(vector_pregunta, fragmento)
    similitudes.append((similitud, i))
    #print(f"Fragmento {i+1}: Similitud = {similitud:.4f}")  

similitudes.sort(reverse=True)

print("Similitudes ordenadas de mayor a menor: ", similitudes[:5])  # Muestra las 5 similitudes más altas

for sim, i in similitudes[:3]:
    print(f"Fragmento {i+1}: Similitud = {sim:.4f}  pagina = {fragmentos[i].metadata['pagina']}  contenido = {fragmentos[i].page_content[:100]}...") 

Similitudes ordenadas de mayor a menor:  [(0.7671638910489283, 3), (0.7627127313429768, 0), (0.7486779978676444, 1), (0.7375273498467518, 6), (0.7133131133553281, 2)]
Fragmento 4: Similitud = 0.7672  pagina = 2  contenido = 5. Productos dañados en el envío
Si el producto llega dañado, el cliente debe reportarlo dentro de l...
Fragmento 1: Similitud = 0.7627  pagina = 1  contenido = TiendaYa — Política de devoluciones y reembolsos
Versión 2.1 · Documento ficticio para uso en el cur...
Fragmento 2: Similitud = 0.7487  pagina = 1  contenido = socio.
2. Plazo para devoluciones
El cliente dispone de 15 días calendario desde la fecha de entrega...


#### 3. Uso de base de datos vectorial ( INGESTA DE DATOS)

Una base de datos vectorial almacena los fragmentos con sus vectores y responde consultas por similitud de forma eficiente, mediante índices especializados. Además persiste el índice en disco y permite filtrar por metadatos.

| Base | Modalidad | Uso típico |
|---|---|---|
| **Chroma** | Local, de código abierto | Desarrollo y proyectos pequeños; se usa en esta sesión |
| Qdrant | Local o gestionada | Producción con filtrado avanzado |

In [ ]:
from langchain_chroma import Chroma

# Borrar base de datos vectorial
Chroma( collection_name="tiendaYa_collection", 
        persist_directory="data/tiendaYa_db",
        embedding_function=embeddings).delete_collection()

# Persistencia de la base de datos de fragmentos en Chroma
base_datos = Chroma.from_documents(
    documents=fragmentos,
    embedding=embeddings,
    persist_directory="data/tiendaYa_db",
    collection_name="tiendaYa_collection",
    collection_metadata={"hnsw:space": "cosine"},
)

print(f"Fragmentos indexados: {base_datos._collection.count()}")


Fragmentos indexados: 25


In [13]:
# Realizando una consulta a la base de datos de fragmentos en Chroma

resultado = base_datos.similarity_search(PREGUNTA, k=3)

for i, doc in enumerate(resultado):
    print(f"Resultado {i+1}: pagina = {doc.metadata['pagina']}  contenido = {doc.page_content[50]}...")




Resultado 1: pagina = 1  contenido = e...
Resultado 2: pagina = 2  contenido = e...
Resultado 3: pagina = 2  contenido = e...


In [14]:
# Realizando una consulta a la base de datos de fragmentos en Chroma

resultado = base_datos.similarity_search_with_score(PREGUNTA, k=3)

for i, (doc, score) in enumerate(resultado):
    print(f"Resultado {i+1}: pagina = {doc.metadata['pagina']}  contenido = {doc.page_content[50]}...  score = {score:.4f}")

Resultado 1: pagina = 1  contenido = e...  score = 0.2027
Resultado 2: pagina = 2  contenido = e...  score = 0.2256
Resultado 3: pagina = 2  contenido = e...  score = 0.2328


###  Realización de Consulta usando RAG

In [15]:
from langchain_core.prompts import ChatPromptTemplate

PREGUNTA = "¿Cuál es la política de devolución de la empresa TiendaYa para productos en oferta?"

# 1. Base de datos vectorial 
retriever = base_datos.as_retriever(search_kwargs={"k": 3})

# 2. Prompt
prompt_rag = ChatPromptTemplate.from_template(
    """ 
    Eres el asistente de TiendaYa. Responde solo con el contexto; si no tiene la información, 
    indica que el documento no lo especifica. Cita la página entre paréntesis.
    
    Contexto:
           {contexto}
           
    Pregunta: 
           {pregunta}

    """
)

# 3. Buscar coincidencias en la base de datos vectorial
fragmentos = retriever.invoke(PREGUNTA)

contexto = ""
for f in fragmentos:
    contexto += f"[página {f.metadata['pagina']}] {f.page_content}\n\n"

# print(contexto)

# 4. Preparar el prompt final

prompt = prompt_rag.invoke({"contexto": contexto, 
                            "pregunta": PREGUNTA})

# print(prompt)

# 5. Usar el LLM

response = llm.invoke(prompt)

print(response.content)


[{'type': 'text', 'text': 'Los productos adquiridos con un descuento igual o mayor al 30 % no admiten devolución por cambio de opinión. Sí admiten devolución si presentan fallas de fábrica o daños durante el transporte, en cuyo caso se aplican los plazos generales de la sección 2 (página 1).', 'extras': {'signature': 'EmAKXgFpFH0TI4Twq7Dvb4laF/mpg3YHri0hr2FWYW4kuQCYsie24IxgMSsmeTsXBrY3yDWYrJRRiMKMAifDqiIg6okefDX/XjJiTxiXEGMDPyCubRyANH8p3UEYK2UxjPE='}}]
